# Feature Extraction, Engineering and Modelling

In [ ]:
# Setup
from datasets import load_dataset
import numpy as np
import pandas as pd
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns

/Users/mattie/Desktop/Masters/Data Science/DataScience2026/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# load token
with open("../token.txt", "r") as f:
    token = f.read().strip()
    
# Load data from hf
ds = load_dataset("MattiWhen/filtered_openfood", token = token)

df = ds["train"].to_pandas()
df.head()

,product_name,brands,nova_group,nutriscore_score,nutriscore_grade,lang,ingredients_original_tags,code,product_name_clean,product_ID
0,"[{'lang': 'main', 'text': 'Peach Sunrise'}, {'...",Once Upon a Farm,1.0,7.0,d,en,"[en:peach-puree, en:coconut-milk, en:apple, en...",0810003517586,Peach Sunrise,Once Upon a Farm_Peach Sunrise
1,"[{'lang': 'main', 'text': 'Ayvar - Roasted Red...",Granny's Secret,3.0,4.0,c,en,"[en:red-bell-pepper, en:sunflower-oil, en:salt...",8606102785030,Ayvar - Roasted Red Pepper Spread,Granny's Secret_Ayvar - Roasted Red Pepper Spread
2,"[{'lang': 'main', 'text': 'Štapići punjeni kik...","Marbo, Pardon, Pepsico",4.0,18.0,d,en,"[en:wheat-flour-min, en:peanut-paste-min, en:p...",8606105235501,Štapići punjeni kikirikijem,"Marbo, Pardon, Pepsico_Štapići punjeni kikirik..."
3,"[{'lang': 'hr', 'text': 'Clipsy max'}, {'lang'...","Marbo,Pepsico",4.0,30.0,e,en,"[en:pellets, en:palm-oil, en:aroma-chicken, en...",8606105235549,Clipsy max,"Marbo,Pepsico_Clipsy max"
4,"[{'lang': 'main', 'text': 'Superseed snack clu...",Manitoba,3.0,17.0,d,en,"[en:hemp-seed, en:tapioca-syrup, en:sunflower-...",0697658201943,Superseed snack clusters,Manitoba_Superseed snack clusters


In [ ]:
df.shape

(624303, 10)

In [ ]:
df = df[~df['nutriscore_grade'].isin(['unknown', 'not-applicable'])]

### Feature Engineering
#### Ingredient parsing

In [ ]:
# ── 1. Check input type ──────────────────────────────────────────────────────
print(df['ingredients_original_tags'].iloc[0])
print(type(df['ingredients_original_tags'].iloc[0]))
print(df['ingredients_original_tags'].apply(type).value_counts())

['en:peach-puree' 'en:coconut-milk' 'en:apple' 'en:strawberry'
 'en:pineapple' 'en:mango' 'en:banana' 'en:orange-juice'
 'en:butternut-squash' 'en:date' 'en:contains' 'en:water' 'en:coconut'
 'en:coconut']
<class 'numpy.ndarray'>
ingredients_original_tags
<class 'numpy.ndarray'>    441921
Name: count, dtype: int64


In [ ]:
# ============================================================
# E NUMBER CLASSIFICATION: NATURAL, SYNTHETIC, AMBIGUOUS
# Source: exploreenumbers.co.uk + EU/FSA classifications
# ============================================================

NATURAL_E = {
    # Colours — plant/mineral origin
    100,   # Curcumin (turmeric)
    101,   # Riboflavin (Vitamin B2)
    140,   # Chlorophylls (plant)
    160,   # Carotenes (carrots/plants) — covers 160a
    162,   # Beetroot red
    163,   # Anthocyanins (fruits/berries)
    170,   # Calcium carbonate (chalk/mineral)
    172,   # Iron oxides (mineral)
    160,   # Carotenes
    161,   # Lutein (161b — egg yolks/vegetables)
    120,   # Cochineal — natural but insect-derived (kept natural, flag vegan separately)

    # Preservatives — naturally occurring
    200,   # Sorbic acid (rowan berries)
    234,   # Nisin (bacterial — natural fermentation)
    260,   # Acetic acid (vinegar)
    261,   # Potassium acetate
    262,   # Sodium acetate
    263,   # Calcium acetate
    270,   # Lactic acid (fermentation)
    290,   # Carbon dioxide
    296,   # Malic acid (natural fruit acid)

    # Antioxidants — vitamins and natural acids
    300,   # Ascorbic acid (Vitamin C)
    301,   # Sodium ascorbate
    302,   # Calcium ascorbate
    306,   # Tocopherols (Vitamin E, from vegetable oils)
    325,   # Sodium lactate
    326,   # Potassium lactate
    327,   # Calcium lactate
    330,   # Citric acid (citrus fruit)
    331,   # Sodium citrates
    332,   # Potassium citrates
    333,   # Calcium citrates
    334,   # Tartaric acid (grapes)
    335,   # Sodium tartrates
    336,   # Potassium tartrates (cream of tartar)
    337,   # Potassium sodium tartrate

    # Thickeners/Emulsifiers — seaweed, plant gums, fruit
    400,   # Alginic acid (seaweed)
    401,   # Sodium alginate (seaweed)
    402,   # Potassium alginate (seaweed)
    403,   # Ammonium alginate (seaweed)
    404,   # Calcium alginate (seaweed)
    406,   # Agar (seaweed)
    410,   # Locust bean gum (carob seeds)
    412,   # Guar gum (guar beans)
    413,   # Tragacanth (tree gum)
    414,   # Gum arabic (tree gum)
    416,   # Karaya gum (tree gum)
    417,   # Tara gum (tara seeds)
    440,   # Pectin (fruit)
    460,   # Cellulose (plant fibre)

    # Acidity regulators / anti-caking — simple minerals
    500,   # Sodium carbonates (baking soda)
    501,   # Potassium carbonates
    504,   # Magnesium carbonates (mineral)
    508,   # Potassium chloride (mineral)
    509,   # Calcium chloride (mineral)
    511,   # Magnesium chloride (mineral)
    515,   # Potassium sulphate (mineral)
    516,   # Calcium sulphate (gypsum/mineral)
    551,   # Silicon dioxide (mineral)
    552,   # Calcium silicate (mineral)
    553,   # Magnesium silicate / Talc (mineral) — covers 553a, 553b
    559,   # Kaolin (mineral)
    574,   # Gluconic acid
    576,   # Sodium gluconate
    577,   # Potassium gluconate
    578,   # Calcium gluconate

    # Flavour enhancers — naturally occurring amino acids
    620,   # Glutamic acid (natural form, found in soy, cheese, tomatoes)

    # Miscellaneous — inert gases, plant waxes, natural sweeteners
    901,   # Beeswax (natural, animal-derived)
    903,   # Carnauba wax (plant)
    904,   # Shellac (insect-derived, natural)
    938,   # Argon (inert gas)
    939,   # Helium (inert gas)
    941,   # Nitrogen (inert gas)
    948,   # Oxygen
    949,   # Hydrogen
    957,   # Thaumatin (katemfe fruit — natural sweetener)
    960,   # Steviol glycosides / Stevia (plant)
    999,   # Quillaia extract (tree bark)

    # Enzymes and natural preservatives
    1103,  # Invertase (enzyme)
    1105,  # Lysozyme (egg white)
    1510,  # Ethanol (fermentation)
}

SYNTHETIC_E = {
    # Colours — synthetic dyes, all require warning labels or are restricted
    102,   # Tartrazine (warning label)
    104,   # Quinoline yellow (warning label)
    110,   # Sunset Yellow FCF (warning label)
    122,   # Carmoisine (warning label)
    123,   # Amaranth (restricted)
    124,   # Ponceau 4R (warning label)
    127,   # Erythrosine
    129,   # Allura Red AC (warning label)
    131,   # Patent Blue V
    132,   # Indigo carmine
    133,   # Brilliant Blue FCF
    142,   # Green S
    150,   # Caramel types b/c/d (chemically treated — 150b, 150c, 150d)
    151,   # Black PN
    154,   # Brown FK
    155,   # Brown HT
    171,   # Titanium dioxide (banned EU, under review UK)
    173,   # Aluminium

    # Preservatives — synthetic
    210,   # Benzoic acid (synthetically produced for food use)
    211,   # Sodium benzoate
    212,   # Potassium benzoate
    213,   # Calcium benzoate
    214,   # Ethyl 4-hydroxybenzoate (paraben)
    215,   # Ethyl 4-hydroxybenzoate sodium salt
    218,   # Methyl 4-hydroxybenzoate (paraben)
    219,   # Sodium salt of E218
    221,   # Sodium sulphite
    222,   # Sodium hydrogen sulphite
    223,   # Sodium metabisulphite
    224,   # Potassium metabisulphite
    226,   # Calcium sulphite
    227,   # Calcium hydrogen sulphite
    228,   # Potassium hydrogen sulphite
    239,   # Hexamine
    242,   # Dimethyl dicarbonate

    # Antioxidants — synthetic
    307,   # Alpha-tocopherol (synthetic vitamin E)
    310,   # Propyl gallate
    311,   # Octyl gallate
    312,   # Dodecyl gallate
    319,   # TBHQ
    320,   # BHA (possible carcinogen)
    321,   # BHT (health concerns)
    385,   # Calcium disodium EDTA

    # Emulsifiers — industrially synthesised
    431,   # Polyoxyethylene stearate
    432,   # Polysorbate 20
    433,   # Polysorbate 80
    434,   # Polysorbate 40
    435,   # Polysorbate 60
    436,   # Polysorbate 65
    472,   # Esters of mono/diglycerides (472a–472f, chemically modified)
    475,   # Polyglycerol esters of fatty acids
    476,   # Polyglycerol polyricinoleate (PGPR)
    477,   # Propylene glycol esters
    481,   # Sodium stearoyl-2-lactylate
    482,   # Calcium stearoyl-2-lactylate
    491,   # Sorbitan monostearate
    492,   # Sorbitan tristearate
    493,   # Sorbitan monolaurate
    494,   # Sorbitan monooleate
    495,   # Sorbitan monopalmitate

    # Acidity regulators — synthetic/industrial chemicals
    507,   # Hydrochloric acid
    512,   # Stannous chloride
    513,   # Sulphuric acid
    520,   # Aluminium sulphate
    521,   # Aluminium sodium sulphate
    522,   # Aluminium potassium sulphate
    523,   # Aluminium ammonium sulphate
    524,   # Sodium hydroxide
    525,   # Potassium hydroxide
    527,   # Ammonium hydroxide
    535,   # Sodium ferrocyanide
    536,   # Potassium ferrocyanide
    538,   # Calcium ferrocyanide
    541,   # Sodium aluminium phosphate

    # Flavour enhancers — synthetic
    637,   # Ethyl maltol

    # Miscellaneous — synthetic gases, waxes, sweeteners
    900,   # Dimethylpolysiloxane
    905,   # Microcrystalline wax
    914,   # Oxidised polyethylene wax
    927,   # Carbamide (927b)
    943,   # Butane / Isobutane (943a, 943b)
    944,   # Propane
    950,   # Acesulfame K
    951,   # Aspartame
    952,   # Cyclamates
    954,   # Saccharin
    955,   # Sucralose
    961,   # Neotame
    962,   # Aspartame-acesulfame salt

    # Additional chemicals — synthetic polymers, solvents
    1200,  # Polydextrose
    1201,  # Polyvinylpyrrolidone
    1202,  # Polyvinylpolypyrrolidone
    1518,  # Glyceryl triacetate
    1520,  # Propylene glycol
}

AMBIGUOUS_E = {
    # Colours — chemically processed or debated origin
    141,   # Copper complexes of chlorophylls (stabilised, chemically modified)
    150,   # Plain caramel E150a (heated sugar — borderline)
    153,   # Carbon black (vegetable carbon but highly processed)
    160,   # E160e Beta-apo-8'-carotenal (can be synthetic)
    161,   # E161g Canthaxanthin (can be synthetic)

    # Preservatives — naturally occurring but synthetically produced at scale
    201,   # Sodium sorbate
    202,   # Potassium sorbate (debated — very widely used)
    203,   # Calcium sorbate
    220,   # Sulphur dioxide (naturally occurring but industrial use)
    234,   # Pimaricin (E235 — natural antifungal, but used as drug-like agent)
    235,   # Pimaricin / Natamycin
    249,   # Potassium nitrite (naturally occurring, ongoing health debate)
    250,   # Sodium nitrite (cured meat — health debate)
    251,   # Sodium nitrate (health debate)
    252,   # Potassium nitrate (health debate)
    280,   # Propionic acid (naturally occurring, synthetically produced)
    281,   # Sodium propionate
    282,   # Calcium propionate (most common bread preservative)
    283,   # Potassium propionate
    284,   # Boric acid
    285,   # Sodium tetraborate
    297,   # Fumaric acid (naturally occurring but synthetically produced)

    # Antioxidants — partially natural, partially processed
    304,   # Ascorbyl palmitate (synthetic derivative of Vitamin C)
    308,   # Gamma-tocopherol
    309,   # Delta-tocopherol
    315,   # Erythorbic acid (similar to Vitamin C, synthetic)
    316,   # Sodium erythorbate
    322,   # Lecithin — KEY AMBIGUOUS (natural from soy/eggs but heavily processed)
    338,   # Phosphoric acid (mineral but industrial scale in cola)
    339,   # Sodium phosphates
    340,   # Potassium phosphates
    341,   # Calcium phosphates
    343,   # Magnesium phosphates
    350,   # Sodium malate
    351,   # Potassium malate
    352,   # Calcium malate
    353,   # Metatartaric acid
    354,   # Calcium tartrate
    355,   # Adipic acid
    356,   # Sodium adipate
    357,   # Potassium adipate
    363,   # Succinic acid
    380,   # Triammonium citrate

    # Thickeners/Emulsifiers — natural source but industrial processing
    405,   # Propane-1,2-diol alginate (chemically modified alginate)
    407,   # Carrageenan — KEY AMBIGUOUS (seaweed but health concerns)
    415,   # Xanthan gum (fermentation — generally natural but industrial)
    418,   # Gellan gum (bacterial fermentation)
    420,   # Sorbitol (naturally occurring, industrially produced)
    421,   # Mannitol (naturally occurring, industrially produced)
    422,   # Glycerol (animal or plant source)
    441,   # Gelatin (animal-derived natural)
    442,   # Ammonium phosphatides
    450,   # Diphosphates
    451,   # Triphosphates
    452,   # Polyphosphates
    461,   # Methyl cellulose (chemically modified cellulose)
    463,   # Hydroxypropyl cellulose (chemically modified)
    464,   # Hydroxypropylmethyl cellulose (chemically modified)
    466,   # Carboxymethyl cellulose (chemically modified)
    470,   # Salts of fatty acids (470a, 470b — animal or plant)
    471,   # Mono- and diglycerides — KEY AMBIGUOUS (animal or plant, industrial)
    473,   # Sucrose esters of fatty acids
    474,   # Sucroglycerides
    483,   # Stearyl tartrate

    # Acidity regulators — mineral but borderline
    503,   # Ammonium carbonates
    514,   # Sodium sulphates
    517,   # Ammonium sulphate
    526,   # Calcium hydroxide (mineral)
    528,   # Magnesium hydroxide
    529,   # Calcium oxide (mineral)
    530,   # Magnesium oxide
    542,   # Edible bone phosphate (animal-derived)
    554,   # Sodium aluminium silicate
    570,   # Fatty acids (animal or plant)
    575,   # Glucono delta-lactone
    579,   # Ferrous gluconate

    # Flavour enhancers — naturally found but commercially synthesised
    621,   # MSG (natural form exists, commercially synthesised)
    622,   # Monopotassium glutamate
    623,   # Calcium glutamate
    624,   # Monoammonium glutamate
    625,   # Magnesium glutamate
    626,   # Guanylic acid
    627,   # Disodium guanylate (sometimes from fish)
    628,   # Dipotassium guanylate
    629,   # Calcium guanylate
    630,   # Inosinic acid (from meat/fish)
    631,   # Disodium inosinate (often from sardines)
    632,   # Dipotassium inosinate
    633,   # Calcium inosinate
    634,   # Calcium 5-ribonucleotides
    635,   # Disodium 5-ribonucleotides (may be from fish)
    636,   # Maltol (naturally found, synthetically produced)
    640,   # Glycine (often from gelatin)

    # Miscellaneous — borderline natural/synthetic
    912,   # Montan acid esters
    920,   # L-cysteine (feathers or synthetic)
    942,   # Nitrous oxide (naturally occurring, industrially produced)
    953,   # Isomalt (derived from sucrose)
    959,   # Neohesperidine DC (from citrus but chemically modified)
    965,   # Maltitol (from starch)
    966,   # Lactitol (from lactose)
    967,   # Xylitol (naturally occurring, industrially produced)
    968,   # Erythritol (fermentation, naturally occurring)

    # Modified starches — natural source, chemically modified
    1204,  # Pullulan (fermentation)
    1401,  # Modified starch
    1402,  # Alkaline modified starch
    1403,  # Bleached starch
    1404,  # Oxidised starch
    1410,  # Monostarch phosphate
    1412,  # Distarch phosphate
    1413,  # Phosphated distarch phosphate
    1414,  # Acetylated distarch phosphate
    1420,  # Acetylated starch
    1422,  # Acetylated distarch adipate
    1440,  # Hydroxypropyl starch
    1442,  # Hydroxypropyl distarch phosphate
    1450,  # Starch sodium octenyl succinate
    1451,  # Acetylated oxidised starch
    1505,  # Triethyl citrate (citric acid derivative)
}

In [ ]:
# Extract E numbers integers from ingredient tags
def extract_e_number(tag):
    """Extract integer from tag, return None if not an E number."""
    match = re.match(r'en:e(\d+)', tag)
    return int(match.group(1)) if match else None

In [ ]:
def engineer_additive_features(tags):
    """Take a list of ingredient tag strings and return dict of additive features."""
    e_nums   = [n for tag in tags if (n := extract_e_number(tag)) is not None]
    natural  = [e for e in e_nums if e in NATURAL_E]
    synthetic = [e for e in e_nums if e in SYNTHETIC_E]
    debated  = [e for e in e_nums if e in AMBIGUOUS_E]

    return {
        'additive_count':           len(e_nums),
        'has_additive':             int(len(e_nums) > 0),
        'natural_additive_count':   len(natural),
        'synthetic_additive_count': len(synthetic),
        'debated_additive_count':   len(debated),
        'has_synthetic':            int(len(synthetic) > 0),
        'has_natural_only':         int(len(synthetic) == 0 and len(natural) > 0),
        'emulsifier_count':         sum(e in range(400, 500) for e in e_nums),
        'preservative_count':       sum(e in range(200, 300) for e in e_nums),
        'sweetener_count':          sum(e in range(950, 970) for e in e_nums),
        'colorant_count':           sum(e in range(100, 200) for e in e_nums),
    }

# apply and expand to columns
additive_features = df['ingredients_original_tags'].apply(engineer_additive_features)
df = pd.concat(
    [df, pd.DataFrame(additive_features.tolist(), index=df.index)],
    axis=1
)

print(df[['additive_count', 'synthetic_additive_count', 
          'natural_additive_count', 'has_synthetic']].describe())

### Data Split
create df_train, df_val, df_test

In [ ]:
# without nova 
missing_nova = df[df["nova_group"].isna()]
# with nova
with_nova = df[df["nova_group"].notna()]

# split with nova into stratified train, test, validation sets
from sklearn.model_selection import train_test_split

# Step 1: split off test (20%)
df_train_val, df_test = train_test_split(
    with_nova, 
    test_size=0.2, 
    stratify=with_nova['nova_group'], 
    random_state=42
)

# Step 2: split remaining 80% into train (70%) and val (10%)
# 0.125 of 80% = 10% of total
df_train, df_val = train_test_split(
    df_train_val, 
    test_size=0.125, 
    stratify=df_train_val['nova_group'], 
    random_state=42
)

print(f"Train: {len(df_train)} ({len(df_train)/len(with_nova):.0%})")
print(f"Val:   {len(df_val)} ({len(df_val)/len(with_nova):.0%})")
print(f"Test:  {len(df_test)} ({len(df_test)/len(with_nova):.0%})")

Train: 272139 (70%)
Val:   38877 (10%)
Test:  77754 (20%)


In [ ]:
# Check stratification by nova group
print(df_train["nova_group"].value_counts())
print(df_test["nova_group"].value_counts())
print(df_val["nova_group"].value_counts())

nova_group
4.0    182141
3.0     54772
1.0     30455
2.0      4771
Name: count, dtype: int64
nova_group
4.0    52040
3.0    15649
1.0     8702
2.0     1363
Name: count, dtype: int64
nova_group
4.0    26020
3.0     7825
1.0     4351
2.0      681
Name: count, dtype: int64


The imbalance across the groups is preserved across all the splits which is good. Means they all reflect the same.

### Feature Extraction

In [ ]:
# ── 1. Find all frequent E numbers in the dataset ──────────────────────────
from collections import Counter

e_counter = Counter()
df['ingredients_original_tags'].apply(
    lambda tags: e_counter.update(
        [n for tag in tags if (n := extract_e_number(tag)) is not None]
    )
)

# keep E numbers appearing in at least 100 products
MIN_COUNT = 100
frequent_e_nums = sorted([e for e, c in e_counter.items() if c >= MIN_COUNT])
print(f"Frequent E numbers to encode individually: {len(frequent_e_nums)}")


In [ ]:

# ── 2. Add individual binary flag columns to df ─────────────────────────────
# do this BEFORE splitting so all splits share the same columns
for e in frequent_e_nums:
    col = f'has_e{e}'
    df[col] = df['ingredients_original_tags'].apply(
        lambda tags: int(any(extract_e_number(t) == e for t in tags))
    )

e_flag_cols = [f'has_e{e}' for e in frequent_e_nums]
print(f"Added {len(e_flag_cols)} individual E number flag columns")


In [ ]:

# ── 3. Define all feature column groups ─────────────────────────────────────
additive_cols = [
    'additive_count',
    'has_additive',
    'natural_additive_count',
    'synthetic_additive_count',
    'debated_additive_count',
    'has_synthetic',
    'has_natural_only',
    'emulsifier_count',
    'preservative_count',
    'sweetener_count',
    'colorant_count',
]

# ── 4. Rebuild splits (in case df changed after adding flag columns) ─────────
missing_nova = df[df["nova_group"].isna()]
with_nova    = df[df["nova_group"].notna()]

df_train_val, df_test = train_test_split(
    with_nova,
    test_size=0.2,
    stratify=with_nova['nova_group'],
    random_state=42
)
df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.125,
    stratify=df_train_val['nova_group'],
    random_state=42
)

print(f"Train: {len(df_train)} ({len(df_train)/len(with_nova):.0%})")
print(f"Val:   {len(df_val)}   ({len(df_val)/len(with_nova):.0%})")
print(f"Test:  {len(df_test)}  ({len(df_test)/len(with_nova):.0%})")


In [ ]:

# ── 5. Nutri-Score grade encoder ─────────────────────────────────────────────
from sklearn.preprocessing import OrdinalEncoder

grade_encoder = OrdinalEncoder(categories=[['a', 'b', 'c', 'd', 'e']])
grade_encoder.fit(df_train[['nutriscore_grade']])


In [ ]:

# ── 6. Build dense feature matrix ────────────────────────────────────────────
def build_feature_matrix(df_split):
    grade     = grade_encoder.transform(df_split[['nutriscore_grade']])  # (n, 1)
    additives = df_split[additive_cols].fillna(0).values                 # (n, 11)
    e_flags   = df_split[e_flag_cols].fillna(0).values                   # (n, k)
    return np.hstack([grade, additives, e_flags])

X_train   = build_feature_matrix(df_train)
X_val     = build_feature_matrix(df_val)
X_test    = build_feature_matrix(df_test)
X_missing = build_feature_matrix(missing_nova)

# store feature names — essential for SHAP plots later
feature_names = ['nutriscore_grade'] + additive_cols + e_flag_cols

print(f"X_train shape: {X_train.shape}")
print(f"  1 grade + {len(additive_cols)} aggregate + {len(e_flag_cols)} E flags "
      f"= {1 + len(additive_cols) + len(e_flag_cols)} features")

### Ordinal Logistic Regression 
Next up, we're doing the ordinal logistic regression.

Notes for the upcoming code:

- We evaluate on X_val, not X_train as we don't want to report training set performance as our result

- QWK is the primary metric here, not accuracy as it penalises predictions that are far off more than ones that are just one step wrong, which matters a lot for ordinal problems like NOVA (i.e. predicting 1 when true is 4 is much worse than predicting 3).

- A QWK above 0.7 would suggest the signal is strong enough to justify moving to a neural network
Thus, the ordinal logistic regression is our baseline — it's fast, interpretable, and tells us whether the features (nutriscore grade, additives, ingredients) actually carry signal for predicting NOVA group.

    - If QWK is already high (0.7+), it means the problem is "easy" for a linear model and a neural network might not add much. If QWK is low, it suggests the relationship is more complex and a neural network could learn patterns the logistic regression can't.

## Wondering what alpha means?

Alpha controls how much the model is penalised for being complex. Think of it as a "simplicity dial":

- High alpha → the model is forced to keep coefficients small and simple. Less risk of overfitting, but might underfit if the signal is subtle.

- Low alpha → the model is free to fit the training data more closely. More expressive, but can overfit.

- alpha=1.0 is just the default starting point — it's a reasonable middle ground. After you see your first results we can tune it if needed, but don't worry about it for now.

## QWK

- QWK stands for Quadratic Weighted Kappa. It measures how well the model's predictions agree with the true labels, but crucially it penalises big mistakes more than small ones.
For example:

- Predicting NOVA 2 when the true answer is NOVA 1 → small penalty (one step off)

- Predicting NOVA 4 when the true answer is NOVA 1 → large penalty (three steps off)

- Plain accuracy doesn't care about this distinction — a wrong answer is a wrong answer. QWK is therefore a much fairer metric for ordinal problems like yours. It ranges from 0 (no better than random) to 1 (perfect).



In [ ]:
import mord
from sklearn.metrics import accuracy_score, cohen_kappa_score


# Target labels (nova_group: 1,2,3,4)
y_train = df_train['nova_group'].astype(int).values - 1  # logisticAT expects 0-indexed classes
y_val   = df_val['nova_group'].astype(int).values - 1

# Initialize and fit ordinal logistic regression
model = mord.LogisticAT(alpha=1.0)  # alpha is regularisation strength
model.fit(X_train, y_train)

# Predict on validation set
y_pred = model.predict(X_val)

# Evaluate
acc = accuracy_score(y_val, y_pred)
qwk = cohen_kappa_score(y_val, y_pred, weights='quadratic')

print(f"Accuracy:              {acc:.4f}")
print(f"Quadratic Weighted Kappa (QWK): {qwk:.4f}")


Accuracy:              0.9302
Quadratic Weighted Kappa (QWK): 0.9321


#### Cross validate alpha over a range

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

for alpha in alphas:
    m = mord.LogisticAT(alpha=alpha)
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy')
    print(f"alpha={alpha:<8} mean acc={scores.mean():.4f}  std={scores.std():.4f}")

alpha=0.001    mean acc=0.9360  std=0.0003
alpha=0.01     mean acc=0.9360  std=0.0004
alpha=0.1      mean acc=0.9349  std=0.0006
alpha=1.0      mean acc=0.9301  std=0.0007
alpha=10.0     mean acc=0.9089  std=0.0011
alpha=100.0    mean acc=0.8566  std=0.0011
